# POS/Cash Feature Engineering

This notebook builds applicant-level features from the cleaned monthly POS/cash data. It creates monthly repayment-progress features, aggregates them up to one row per previous loan, then to one row per applicant, and fills in zeros for applicants with no POS/cash history at all.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
input_path = project_root / "data" / "interim" / "pos_cash_balance_clean.pkl"
application_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "pos_cash_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [input_path, application_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Clean input:", input_path)
print("Feature output:", output_path)

Clean input: /Users/taranveersingh/A-MRP/data/interim/pos_cash_balance_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/pos_cash_features.pkl


## Load cleaned data and split information


In [7]:
pos = pd.read_pickle(input_path)
application_target = pd.read_pickle(application_path)[["SK_ID_CURR", "TARGET"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
project_ids = training_id_set.union(test_id_set)
print("Clean monthly POS/cash shape:", pos.shape)
print("Previous loans represented:", pos["SK_ID_PREV"].nunique())
print("Applicants with POS/cash history:", pos["SK_ID_CURR"].nunique())

Clean monthly POS/cash shape: (8543375, 14)
Previous loans represented: 800337
Applicants with POS/cash history: 289444


Most applicants (289,444 of 307,511) have some POS/cash history.


## Create monthly repayment-progress features


In [10]:
def safe_ratio(numerator, denominator):
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

pos["POS_REMAINING_INSTALLMENT_RATIO"] = safe_ratio(pos["CNT_INSTALMENT_FUTURE"], pos["CNT_INSTALMENT"])
pos["POS_COMPLETION_RATIO"] = 1 - pos["POS_REMAINING_INSTALLMENT_RATIO"]
pos["POS_COMPLETED_INSTALLMENTS"] = pos["CNT_INSTALMENT"] - pos["CNT_INSTALMENT_FUTURE"]
pos["POS_IS_DELINQUENT"] = pos["SK_DPD"].gt(0).astype("int8")
pos["POS_IS_SEVERELY_DELINQUENT"] = pos["SK_DPD"].gt(30).astype("int8")
pos["POS_IS_ACTIVE"] = pos["NAME_CONTRACT_STATUS"].eq("Active").astype("int8")
pos["POS_IS_COMPLETED"] = pos["NAME_CONTRACT_STATUS"].eq("Completed").astype("int8")
pos["POS_SCHEDULE_AVAILABLE"] = pos[["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].notna().all(axis=1).astype("int8")
pos["POS_RECENT_12M"] = pos["MONTHS_BALANCE"].ge(-12).astype("int8")
print("Monthly repayment-progress features created: 9")

Monthly repayment-progress features created: 9


These track how much of each loan's repayment schedule has been completed, and whether the loan is overdue.


## Aggregate monthly history to previous-loan level


In [13]:
account_features = pos.groupby(["SK_ID_CURR", "SK_ID_PREV"]).agg(
    POS_MONTH_COUNT=("MONTHS_BALANCE", "count"),
    POS_OLDEST_MONTH=("MONTHS_BALANCE", "min"),
    POS_LATEST_MONTH=("MONTHS_BALANCE", "max"),
    POS_TOTAL_INSTALLMENTS_MEAN=("CNT_INSTALMENT", "mean"),
    POS_TOTAL_INSTALLMENTS_MAX=("CNT_INSTALMENT", "max"),
    POS_FUTURE_INSTALLMENTS_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_FUTURE_INSTALLMENTS_MIN=("CNT_INSTALMENT_FUTURE", "min"),
    POS_COMPLETION_RATIO_MEAN=("POS_COMPLETION_RATIO", "mean"),
    POS_COMPLETION_RATIO_MAX=("POS_COMPLETION_RATIO", "max"),
    POS_DELINQUENT_MONTH_COUNT=("POS_IS_DELINQUENT", "sum"),
    POS_DELINQUENT_MONTH_RATE=("POS_IS_DELINQUENT", "mean"),
    POS_SEVERE_DELINQUENT_MONTH_COUNT=("POS_IS_SEVERELY_DELINQUENT", "sum"),
    POS_DPD_MAX=("SK_DPD", "max"),
    POS_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    POS_ACTIVE_MONTH_RATE=("POS_IS_ACTIVE", "mean"),
    POS_COMPLETED_MONTH_RATE=("POS_IS_COMPLETED", "mean"),
    POS_SCHEDULE_AVAILABLE_RATE=("POS_SCHEDULE_AVAILABLE", "mean"),
    POS_SCHEDULE_MISSING_COUNT=("POS_SCHEDULE_MISSING", "sum"),
    POS_FUTURE_ABOVE_TOTAL_COUNT=("POS_FUTURE_ABOVE_TOTAL", "sum"),
    POS_COMPLETED_WITH_REMAINING_COUNT=("POS_COMPLETED_WITH_REMAINING", "sum"),
    POS_ACTIVE_WITH_ZERO_FUTURE_COUNT=("POS_ACTIVE_WITH_ZERO_FUTURE", "sum"),
    POS_RECORD_MISSING_RATE_MEAN=("POS_RECORD_MISSING_RATE", "mean"),
).reset_index()
print("Previous POS/cash loans summarized:", len(account_features))
print("Account feature shape:", account_features.shape)

Previous POS/cash loans summarized: 800337
Account feature shape: (800337, 24)


Each previous loan's monthly records are summarized here, total months, completion ratio, and delinquency counts.


## Add latest account state


In [16]:
latest_rows = pos.sort_values("MONTHS_BALANCE").groupby(["SK_ID_CURR", "SK_ID_PREV"], as_index=False).tail(1)
latest_features = latest_rows[[
    "SK_ID_CURR", "SK_ID_PREV", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "POS_COMPLETION_RATIO", "SK_DPD", "SK_DPD_DEF", "POS_IS_ACTIVE", "POS_IS_COMPLETED"
]].rename(columns={
    "CNT_INSTALMENT": "POS_LATEST_TOTAL_INSTALLMENTS",
    "CNT_INSTALMENT_FUTURE": "POS_LATEST_FUTURE_INSTALLMENTS",
    "POS_COMPLETION_RATIO": "POS_LATEST_COMPLETION_RATIO",
    "SK_DPD": "POS_LATEST_DPD",
    "SK_DPD_DEF": "POS_LATEST_DPD_DEF",
    "POS_IS_ACTIVE": "POS_LATEST_IS_ACTIVE",
    "POS_IS_COMPLETED": "POS_LATEST_IS_COMPLETED",
})
account_features = account_features.merge(
    latest_features, on=["SK_ID_CURR", "SK_ID_PREV"], how="left", validate="one_to_one"
)
print("Latest-state features added: 7")

Latest-state features added: 7


This grabs the values from each loan's most recent monthly record, like the latest completion ratio and days overdue, since the current state can matter more than the average.


## Add recent 12-month behaviour


In [19]:
recent_pos = pos.loc[pos["POS_RECENT_12M"].eq(1)]
recent_account = recent_pos.groupby(["SK_ID_CURR", "SK_ID_PREV"]).agg(
    POS_RECENT_12M_COUNT=("MONTHS_BALANCE", "count"),
    POS_RECENT_12M_DELINQUENT_COUNT=("POS_IS_DELINQUENT", "sum"),
    POS_RECENT_12M_DELINQUENT_RATE=("POS_IS_DELINQUENT", "mean"),
    POS_RECENT_12M_DPD_MAX=("SK_DPD", "max"),
    POS_RECENT_12M_COMPLETION_RATIO_MEAN=("POS_COMPLETION_RATIO", "mean"),
).reset_index()
account_features = account_features.merge(
    recent_account, on=["SK_ID_CURR", "SK_ID_PREV"], how="left", validate="one_to_one"
)
recent_columns = [c for c in recent_account.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
account_features[recent_columns] = account_features[recent_columns].fillna(0)
del recent_pos, recent_account, latest_rows, latest_features, pos
print("Account feature shape after latest and recent features:", account_features.shape)

Account feature shape after latest and recent features: (800337, 36)


Same kind of summary as before, but limited to the most recent 12 months only.


## Aggregate previous loans to applicant level


In [22]:
aggregation_rules = {}
for column in [c for c in account_features.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]:
    if column.endswith("_COUNT"):
        aggregation_rules[column] = "sum"
    elif column == "POS_OLDEST_MONTH" or column.endswith("_MIN"):
        aggregation_rules[column] = "min"
    elif column in ["POS_LATEST_MONTH", "POS_LATEST_DPD", "POS_LATEST_DPD_DEF"] or column.endswith("_MAX"):
        aggregation_rules[column] = "max"
    else:
        aggregation_rules[column] = "mean"

pos_features = account_features.groupby("SK_ID_CURR").agg(aggregation_rules).reset_index()
loan_counts = account_features.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename("POS_PREVIOUS_LOAN_COUNT").reset_index()
pos_features = pos_features.merge(loan_counts, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Applicants with POS/cash history summarized:", len(pos_features))
print("Applicant history feature shape:", pos_features.shape)

Applicants with POS/cash history summarized: 289444
Applicant history feature shape: (289444, 36)


Applicants with more than one previous POS/cash loan have their loan-level summaries combined here into a single row per applicant.


## Add all applicants and encode structural absence


In [25]:
all_applicants = application_target[["SK_ID_CURR"]].copy()
all_applicants["POS_HISTORY_AVAILABLE"] = all_applicants["SK_ID_CURR"].isin(set(pos_features["SK_ID_CURR"])).astype("int8")
pos_features = all_applicants.merge(pos_features, on="SK_ID_CURR", how="left", validate="one_to_one")
feature_columns_before_rules = [c for c in pos_features.columns if c != "SK_ID_CURR"]
pos_features[feature_columns_before_rules] = pos_features[feature_columns_before_rules].fillna(0)
print("All applicants included:", len(pos_features))
print("Applicants with POS/cash history:", int(pos_features["POS_HISTORY_AVAILABLE"].sum()))
print("Applicants without POS/cash history:", int(pos_features["POS_HISTORY_AVAILABLE"].eq(0).sum()))

All applicants included: 307511
Applicants with POS/cash history: 289444
Applicants without POS/cash history: 18067


Applicants with no POS/cash history at all are added back in here, with their feature values set to 0 rather than left missing, since that is still meaningful information about the applicant.


## Apply training-only missingness and constant-feature rules


In [28]:
MISSING_THRESHOLD = 0.50
training_base = pd.DataFrame({"SK_ID_CURR": training_ids}).merge(
    application_target, on="SK_ID_CURR", how="left", validate="one_to_one"
).merge(pos_features, on="SK_ID_CURR", how="left", validate="one_to_one")
decision_rows = []
for feature in [c for c in pos_features.columns if c != "SK_ID_CURR"]:
    series = training_base[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    correlation = series.corr(training_base["TARGET"]) if unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for global cross-validated feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-applicant missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set"
    decision_rows.append({
        "feature": feature, "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
pos_features = pos_features.drop(columns=removed_features)
print("Features removed:", removed_features)
print("Features retained:", pos_features.shape[1] - 1)
feature_decisions.round(5)

Features removed: []
Features retained: 36


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,decision,reason
0,POS_OLDEST_MONTH,0,0.0,97,0.04607,0.04607,Keep,Retain for global cross-validated feature sele...
1,POS_FUTURE_INSTALLMENTS_MEAN,0,0.0,66712,0.03567,0.03567,Keep,Retain for global cross-validated feature sele...
2,POS_DELINQUENT_MONTH_RATE,0,0.0,6133,0.03428,0.03428,Keep,Retain for global cross-validated feature sele...
3,POS_PREVIOUS_LOAN_COUNT,0,0.0,25,-0.03367,0.03367,Keep,Retain for global cross-validated feature sele...
4,POS_RECENT_12M_DELINQUENT_COUNT,0,0.0,20,0.03188,0.03188,Keep,Retain for global cross-validated feature sele...
5,POS_RECENT_12M_DELINQUENT_RATE,0,0.0,409,0.03053,0.03053,Keep,Retain for global cross-validated feature sele...
6,POS_MONTH_COUNT,0,0.0,222,-0.03051,0.03051,Keep,Retain for global cross-validated feature sele...
7,POS_TOTAL_INSTALLMENTS_MEAN,0,0.0,44505,0.02851,0.02851,Keep,Retain for global cross-validated feature sele...
8,POS_RECENT_12M_COMPLETION_RATIO_MEAN,0,0.0,21674,0.02589,0.02589,Keep,Retain for global cross-validated feature sele...
9,POS_LATEST_FUTURE_INSTALLMENTS,0,0.0,748,0.02052,0.02052,Keep,Retain for global cross-validated feature sele...


No POS/cash features needed to be removed, all 36 stayed within the limits.


## Validate the applicant-level feature table


In [31]:
numeric_columns = pos_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(pos_features[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = feature_decisions.loc[feature_decisions["decision"] == "Keep"]
validation_checks = pd.DataFrame([
    {"check": "One row per project applicant", "passed": pos_features["SK_ID_CURR"].is_unique and len(pos_features) == len(application_target)},
    {"check": "All project applicants included", "passed": set(pos_features["SK_ID_CURR"]) == project_ids},
    {"check": "No TARGET in output", "passed": "TARGET" not in pos_features.columns},
    {"check": "No previous-loan ID in output", "passed": "SK_ID_PREV" not in pos_features.columns},
    {"check": "History availability retained", "passed": "POS_HISTORY_AVAILABLE" in pos_features.columns},
    {"check": "No missing values after structural encoding", "passed": pos_features.isna().sum().sum() == 0},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from feature decisions", "passed": not training_base["SK_ID_CURR"].isin(test_id_set).any()},
])
assert validation_checks["passed"].all(), "At least one POS/cash feature-engineering check failed."
validation_checks

,check,passed
0,One row per project applicant,True
1,All project applicants included,True
2,No TARGET in output,True
3,No previous-loan ID in output,True
4,History availability retained,True
5,No missing values after structural encoding,True
6,No infinite numerical values,True
7,No retained feature reaches 50 percent trainin...,True
8,Final test excluded from feature decisions,True


All checks passed.


## Save features and audit reports


In [34]:
pos_features.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "pos_cash_engineered_feature_decisions.csv", index=False)
validation_checks.to_csv(audit_folder / "pos_cash_feature_engineering_validation.csv", index=False)
print("POS/cash feature table saved:", output_path)
print("Output rows:", len(pos_features))
print("Output columns:", pos_features.shape[1])
print("Applicants with history:", int(pos_features["POS_HISTORY_AVAILABLE"].sum()))
print("Remaining numerical missing values:", int(pos_features.select_dtypes(include="number").isna().sum().sum()))

POS/cash feature table saved: /Users/taranveersingh/A-MRP/data/features/pos_cash_features.pkl
Output rows: 307511
Output columns: 37
Applicants with history: 289444
Remaining numerical missing values: 0


## Main feature engineering results

This notebook built 36 applicant-level features from the cleaned monthly POS/cash data, covering repayment progress, latest loan state, and recent 12-month behaviour, plus a flag for whether the applicant has any POS/cash history at all.

All checks passed, and no features were removed. The feature table has 37 columns for all 307,511 applicants, with zero remaining missing values since applicants without POS/cash history are filled with 0. This was the last of the seven source tables to go through feature engineering.
